In [ ]:
# --- [ 1단계: 모든 흔적 지우기 및 con 재생성 ] ---
import os
import duckdb
from dotenv import load_dotenv , find_dotenv
load_dotenv(find_dotenv())
endpoint = os.getenv('MINIO_ENDPOINT')
access_key = os.getenv('MINIO_ACCESS_KEY')
secret_key = os.getenv('MINIO_SECRET_KEY')
# 아예 새 그릇을 꺼냅니다 (기존 con이 있다면 닫힘)
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
# --- [ 2단계: 'Secret' 저장소에 직접 각인하기 ] ---
# 기존에 꼬여있던 낱개 SET 방식이 아니라, 하나의 '비밀번호 팩'을 만드는 방식입니다.
con.execute(f"""
    CREATE OR REPLACE SECRET my_minio_secret (
        TYPE S3,
        KEY_ID '{access_key}',
        SECRET '{secret_key}',
        ENDPOINT '{endpoint}',
        URL_STYLE 'path',
        USE_SSL 'false',
        REGION 'us-east-1'
    );
""")
# --- [ 3단계: %sql 매직 커맨드 초기화 ] ---
%load_ext sql
%sql con 
# 잘 되는지 확인용 (이게 에러 안 나면 성공입니다)
print("✅ [강제 초기화 완료] 이제 찌꺼기 없는 깨끗한 상태입니다. 쿼리를 날려보세요!")

✅ [강제 초기화 완료] 이제 찌꺼기 없는 깨끗한 상태입니다. 쿼리를 날려보세요!


In [ ]:
%%sql

select *
from 's3://petroleum-project/national_avg/*/*.parquet'
where part_dt >= '20260124'
and prodnm = '휘발유'

In [ ]:
%%sql

select 
part_dt 
,katec_x 
, katec_y 
, count(distinct uni_cd) as gasstation_cnt
from 's3://petroleum-project/coord_validation_logs/part_dt=20260129/data.parquet'
where part_dt >= '20260129'
group by 1,2,3


In [ ]:
# %%sql 없이, 순수 파이썬 코드로 파일 내용을 읽어봅니다.
try:
    # 1. 일단 파일 하나만 콕 찝어서 전체 데이터를 가져와봅니다.
    path = "s3://petroleum-project/coord_validation_logs/part_dt=20260129/data.parquet"
    test_df = con.sql(f"SELECT * FROM '{path}'").df()
    
    print("✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.")
    display(test_df)
    
except Exception as e:
    print(f"❌ 엔진 레벨에서 에러가 발생했습니다: {e}")

✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.


,part_dt,katec_x,katec_y,uni_cd,brand_cd,station_name,station_x,station_y,fuel_type,price,collect_time
0,20260129,239534.420964,476124.618520,A0015034,SKE,학암포주유소,241063.60760,475627.22940,B027,1760,2026-01-29 22:32:45
1,20260129,248149.485547,475966.537112,A0014891,SOL,이원주유소,246545.79840,473537.83570,B027,1750,2026-01-29 22:32:45
2,20260129,256765.319674,475817.168258,A0017070,SKE,파워주유소,260980.62097,475927.56494,B027,1698,2026-01-29 22:32:45
3,20260129,256765.319674,475817.168258,A0009246,HDO,지곡현대주유소,261302.00000,476954.00000,B027,1698,2026-01-29 22:32:45
4,20260129,256765.319674,475817.168258,A0016967,GSC,신성주유소,260643.30470,474186.54280,B027,1735,2026-01-29 22:32:45
...,...,...,...,...,...,...,...,...,...,...,...
3181,20260129,352411.174466,627703.976236,A0004302,HDO,행복주유소,350949.10000,626534.50000,B027,1674,2026-01-29 22:32:45
3182,20260129,352411.174466,627703.976236,A0033657,NHO,김화농협주유소,351016.49412,626200.31686,B027,1675,2026-01-29 22:32:45
3183,20260129,352411.174466,627703.976236,A0004434,SOL,근남주유소,353942.36580,626469.13500,B027,1699,2026-01-29 22:32:45
3184,20260129,352411.174466,627703.976236,A0004715,GSC,와수주유소,352038.00000,627622.00000,B027,1699,2026-01-29 22:32:45


In [ ]:
query = """
SELECT 
    m.lat,
    m.lon,
    m.katec_x, 
    m.katec_y, 
    COUNT(g.uni_cd) as station_cnt
FROM 'geographic_master_table.parquet' m
LEFT JOIN 's3://petroleum-project/coord_validation_logs/part_dt=20260129/data.parquet' g
  ON m.katec_x = g.katec_x AND m.katec_y = g.katec_y
GROUP BY 1, 2,3,4
ORDER BY 5 ASC
"""
# 쿼리 실행 및 결과 보기
df_result = con.sql(query).df()
print("✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:")
zero_stations = df_result[df_result['station_cnt'] == 0]
display(df_result.sort_values(by='station_cnt' , ascending=False))
display(zero_stations)
print(f"총 {len(zero_stations)}개의 좌표가 주유소가 없습니다. (전체 {len(df_result)}개 중)")

✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:


,lat,lon,katec_x,katec_y,station_cnt
359,126.68316,37.48261,283553.741485,543400.331300,89
358,126.87643,37.48261,300644.951593,543178.805158,71
357,127.06969,37.25288,317483.742690,517500.289373,68
356,126.77980,37.48261,292099.828892,543285.174663,65
355,126.87643,37.55919,300746.461427,551676.889126,59
...,...,...,...,...,...
92,126.58653,38.01865,275907.109549,603012.532185,0
93,126.29663,37.32946,249064.087410,526952.832397,0
94,126.39327,37.02315,257051.847410,492811.598620,0
95,126.39327,37.86550,258646.050462,586290.975175,0


,lat,lon,katec_x,katec_y,station_cnt
0,126.77980,38.24838,293210.888676,628268.190676,0
1,126.48990,37.17631,265919.695357,509666.483519,0
2,127.55286,37.94207,360704.259252,593665.618623,0
3,127.84276,38.32495,386253.405888,636073.523997,0
4,126.48990,38.24838,267838.251219,628642.515230,0
...,...,...,...,...,...
124,127.45623,38.32495,352461.145482,636201.719435,0
125,126.20000,38.01865,241970.020259,603598.853769,0
126,126.87643,38.09523,301461.973542,611164.227729,0
127,126.20000,37.63577,241151.034715,561105.331005,0


총 129개의 좌표가 주유소가 없습니다. (전체 360개 중)


In [ ]:
# %%sql 없이, 순수 파이썬 코드로 파일 내용을 읽어봅니다.
try:
    # 1. 일단 파일 하나만 콕 찝어서 전체 데이터를 가져와봅니다.
    path = "s3://petroleum-project/coord_validation_logs/part_dt=20260130/data.parquet"
    test_df = con.sql(f"SELECT * FROM '{path}'").df()
    
    print("✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.")
    display(test_df)
    
except Exception as e:
    print(f"❌ 엔진 레벨에서 에러가 발생했습니다: {e}")

✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.


,part_dt,katec_x,katec_y,uni_cd,brand_cd,station_name,station_x,station_y,fuel_type,price,collect_time
0,20260130,239534.420964,476124.618520,A0015034,SKE,학암포주유소,241063.60760,475627.22940,B027,1760,2026-01-30 22:51:48
1,20260130,248149.485547,475966.537112,A0014891,SOL,이원주유소,246545.79840,473537.83570,B027,1750,2026-01-30 22:51:48
2,20260130,256765.319674,475817.168258,A0009246,HDO,지곡현대주유소,261302.00000,476954.00000,B027,1698,2026-01-30 22:51:48
3,20260130,256765.319674,475817.168258,A0017070,SKE,파워주유소,260980.62097,475927.56494,B027,1698,2026-01-30 22:51:48
4,20260130,256765.319674,475817.168258,A0016967,GSC,신성주유소,260643.30470,474186.54280,B027,1735,2026-01-30 22:51:48
...,...,...,...,...,...,...,...,...,...,...,...
3311,20260130,352377.267376,621929.746390,A0004434,SOL,근남주유소,353942.36580,626469.13500,B027,1699,2026-01-30 22:51:48
3312,20260130,352377.267376,621929.746390,A0000832,SOL,화강주유소,350870.31400,626289.87600,B027,1719,2026-01-30 22:51:48
3313,20260130,352377.267376,621929.746390,A0011861,SKE,송동주유소,349526.83280,622179.42290,B027,1719,2026-01-30 22:51:48
3314,20260130,369302.813357,621848.066867,A0010455,SKE,상서주유소,371003.58850,621676.57580,B027,1792,2026-01-30 22:51:48


In [ ]:
query = """
SELECT 
    m.lat,
    m.lon,
    m.katec_x, 
    m.katec_y, 
    COUNT(g.uni_cd) as station_cnt
FROM 'geographic_master_table_hex.parquet' m
LEFT JOIN 's3://petroleum-project/coord_validation_logs/part_dt=20260130/data.parquet' g
  ON m.katec_x = g.katec_x AND m.katec_y = g.katec_y
GROUP BY 1, 2,3,4
ORDER BY 5 ASC
"""
# 쿼리 실행 및 결과 보기
df_result = con.sql(query).df()
print("✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:")
zero_stations = df_result[df_result['station_cnt'] == 0]
display(df_result.sort_values(by='station_cnt' , ascending=False))
display(zero_stations)
print(f"총 {len(zero_stations)}개의 좌표가 주유소가 없습니다. (전체 {len(df_result)}개 중)")

✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:


,lat,lon,katec_x,katec_y,station_cnt
413,126.68316,37.46686,283529.297273,541652.533730,83
412,126.87643,37.46686,300624.096410,541431.040503,60
411,127.06969,37.26790,317500.128336,519166.961324,58
410,126.87643,37.53317,300711.950924,548789.437383,57
409,126.77980,37.53317,292172.592685,548895.857411,55
...,...,...,...,...,...
100,126.58653,37.13527,274431.998646,504980.097813,0
99,126.29663,38.13003,250680.924848,615800.746359,0
98,126.48990,38.26266,267864.121186,630227.415051,0
97,127.16633,38.32898,327120.455896,636837.910728,0


,lat,lon,katec_x,katec_y,station_cnt
0,127.35959,38.32898,344015.460105,636703.118063,0
1,126.20000,36.93632,239673.296136,483484.036281,0
2,126.58653,38.19635,276207.427891,622734.323478,0
3,126.20000,37.13527,240091.192132,505561.624120,0
4,126.29663,38.26266,250951.617497,630521.195598,0
...,...,...,...,...,...
146,126.48990,37.20159,265964.396560,512471.772501,0
147,126.48990,37.86476,267146.364392,586066.876718,0
148,127.84276,37.20159,386043.902378,511415.245645,0
149,127.35959,38.26266,343964.475114,629342.879032,0


총 151개의 좌표가 주유소가 없습니다. (전체 414개 중)


In [ ]:
# %%sql 없이, 순수 파이썬 코드로 파일 내용을 읽어봅니다.
try:
    # 1. 일단 파일 하나만 콕 찝어서 전체 데이터를 가져와봅니다.
    path = "s3://petroleum-project/coord_validation_logs/part_dt=20260128/data.parquet"
    test_df = con.sql(f"SELECT * FROM '{path}'").df()
    
    print("✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.")
    display(test_df)
    
except Exception as e:
    print(f"❌ 엔진 레벨에서 에러가 발생했습니다: {e}")

✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.


,part_dt,katec_x,katec_y,uni_cd,brand_cd,station_name,station_x,station_y,fuel_type,price,collect_time
0,20260128,239534.420964,476124.618520,A0015034,SKE,학암포주유소,241063.60760,475627.22940,B027,1760,2026-01-30 23:06:07
1,20260128,248311.746774,475963.643434,A0014891,SOL,이원주유소,246545.79840,473537.83570,B027,1750,2026-01-30 23:06:07
2,20260128,257088.946091,475811.727741,A0009246,HDO,지곡현대주유소,261302.00000,476954.00000,B027,1698,2026-01-30 23:06:07
3,20260128,257088.946091,475811.727741,A0017070,SKE,파워주유소,260980.62097,475927.56494,B027,1698,2026-01-30 23:06:07
4,20260128,257088.946091,475811.727741,A0016967,GSC,신성주유소,260643.30470,474186.54280,B027,1735,2026-01-30 23:06:07
...,...,...,...,...,...,...,...,...,...,...,...
3301,20260128,354463.198521,624687.883372,A0004302,HDO,행복주유소,350949.10000,626534.50000,B027,1674,2026-01-30 23:06:07
3302,20260128,354463.198521,624687.883372,A0033657,NHO,김화농협주유소,351016.49412,626200.31686,B027,1675,2026-01-30 23:06:07
3303,20260128,354463.198521,624687.883372,A0004434,SOL,근남주유소,353942.36580,626469.13500,B027,1699,2026-01-30 23:06:07
3304,20260128,354463.198521,624687.883372,A0004715,GSC,와수주유소,352038.00000,627622.00000,B027,1699,2026-01-30 23:06:07


In [ ]:
query = """
SELECT 
    m.lat,
    m.lon,
    m.katec_x, 
    m.katec_y, 
    COUNT(g.uni_cd) as station_cnt
FROM 'geographic_master_table_hex.parquet' m
LEFT JOIN 's3://petroleum-project/coord_validation_logs/part_dt=20260128/data.parquet' g
  ON m.katec_x = g.katec_x AND m.katec_y = g.katec_y
GROUP BY 1, 2,3,4
ORDER BY 5 ASC
"""
# 쿼리 실행 및 결과 보기
df_result = con.sql(query).df()
print("✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:")
zero_stations = df_result[df_result['station_cnt'] == 0]
display(df_result.sort_values(by='station_cnt' , ascending=False))
display(zero_stations)
print(f"총 {len(zero_stations)}개의 좌표가 주유소가 없습니다. (전체 {len(df_result)}개 중)")

✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:


,lat,lon,katec_x,katec_y,station_cnt
395,126.69226,37.47809,284351.510572,542887.523236,76
394,126.88916,37.47809,301764.759168,542663.865567,65
393,127.08607,37.27539,318960.769061,519983.920703,61
392,126.98761,37.27539,310229.974355,520072.818356,60
391,126.88916,37.54566,301853.292450,550162.090187,55
...,...,...,...,...,...
98,126.59381,37.81592,276206.875392,580503.832418,0
100,127.87368,38.28888,388951.095153,632066.351435,0
101,127.87368,37.27539,388799.184743,519599.931009,0
102,126.20000,37.41052,240672.543465,536107.364440,0


,lat,lon,katec_x,katec_y,station_cnt
0,126.29845,37.88348,250339.949911,588434.331614,0
1,126.79071,38.22131,294126.491269,625251.326082,0
2,126.59381,38.15375,276773.312591,617996.669672,0
3,126.39690,37.34296,257975.096960,528295.445663,0
4,127.67678,38.22131,371702.505972,624609.384389,0
...,...,...,...,...,...
133,126.59381,37.95105,276432.929921,595500.502636,0
134,126.20000,37.88348,241680.079547,588596.873256,0
135,126.59381,37.34296,275421.123836,528017.473961,0
136,127.18452,38.08618,328473.427671,609877.758000,0


총 138개의 좌표가 주유소가 없습니다. (전체 396개 중)


In [ ]:
# %%sql 없이, 순수 파이썬 코드로 파일 내용을 읽어봅니다.
try:
    # 1. 일단 파일 하나만 콕 찝어서 전체 데이터를 가져와봅니다.
    path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
    test_df = con.sql(f"SELECT part_dt , count(distinct uni_cd) FROM '{path}' group by 1 order by 1").df()
    
    print("✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.")
    display(test_df)
    
except Exception as e:
    print(f"❌ 엔진 레벨에서 에러가 발생했습니다: {e}")

✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.


,part_dt,count(DISTINCT uni_cd)
0,20260127,3387
1,20260128,3306
2,20260129,3186
3,20260130,3316


In [ ]:
query = """
SELECT 
    m.lat,
    m.lon,
    m.katec_x, 
    m.katec_y, 
    COUNT(distinct g.uni_cd) as station_cnt
FROM 'geographic_master_table_hex.parquet' m
LEFT JOIN 's3://petroleum-project/coord_validation_logs/part_dt=20260127/data.parquet' g
  ON m.katec_x = g.katec_x AND m.katec_y = g.katec_y
GROUP BY 1, 2,3,4
ORDER BY 5 ASC
"""
# 쿼리 실행 및 결과 보기
df_result = con.sql(query).df()
print("✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:")
zero_stations = df_result[df_result['station_cnt'] == 0]
display(df_result.sort_values(by='station_cnt' , ascending=False))
display(zero_stations)
print(f"총 {len(zero_stations)}개의 좌표가 주유소가 없습니다. (전체 {len(df_result)}개 중)")

✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:


,lat,lon,katec_x,katec_y,station_cnt
455,126.74569,37.49416,289100.495705,544606.530854,64
454,126.83664,37.49416,297142.117523,544503.244595,60
453,126.65474,37.49416,281058.800812,544717.591740,60
452,126.65474,37.43175,280959.874863,537791.866629,59
451,127.01854,37.24450,312937.102147,516616.236599,58
...,...,...,...,...,...
110,126.20000,36.99483,239795.995870,489976.853057,0
109,127.38233,38.30557,345986.039424,634091.517422,0
108,126.47285,37.11966,264304.538819,503404.386393,0
107,127.74612,38.18074,377760.806526,620088.110653,0


,lat,lon,katec_x,katec_y,station_cnt
0,126.65474,37.86866,281655.388998,586277.834001,0
1,126.29095,37.93108,249777.081368,593729.227969,0
2,126.38190,38.18074,258254.487763,621295.073063,0
3,126.20000,37.74383,241381.457467,573098.001266,0
4,126.92759,38.24316,306138.798965,627528.635692,0
...,...,...,...,...,...
162,126.38190,37.05725,256104.592494,496612.744437,0
163,126.29095,36.93242,247767.180277,482902.132451,0
164,127.83707,37.24450,385547.066873,516177.349131,0
165,126.29095,37.61899,249144.023896,559092.957163,0


총 167개의 좌표가 주유소가 없습니다. (전체 456개 중)


In [ ]:
# %%sql 없이, 순수 파이썬 코드로 파일 내용을 읽어봅니다.
try:
    # 1. 일단 파일 하나만 콕 찝어서 전체 데이터를 가져와봅니다.
    path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
    test_df = con.sql(f"""
    SELECT uni_cd
    , station_name
    , max(case when part_dt=20260126 then 1 else 0 end ) as is_075
    , max(case when part_dt=20260127 then 1 else 0 end ) as is_080
    , max(case when part_dt=20260128 then 1 else 0 end ) as is_085
    FROM '{path}' 
    group by 1,2 
    order by 
    """
    
    ).df()
    
    print("✅ 엔진은 정상입니다! 파일도 잘 읽힙니다.")
    display(test_df)
    
except Exception as e:
    print(f"❌ 엔진 레벨에서 에러가 발생했습니다: {e}")

❌ 엔진 레벨에서 에러가 발생했습니다: Parser Error: syntax error at or near "and"

LINE 9:     having is_075 =  and is_080=0
                             ^


In [ ]:
query = """
SELECT 
    m.lat,
    m.lon,
    m.katec_x, 
    m.katec_y, 
    COUNT(distinct g.uni_cd) as station_cnt
FROM 'geographic_master_table_hex.parquet' m
LEFT JOIN 's3://petroleum-project/coord_validation_logs/part_dt=20260126/data.parquet' g
  ON m.katec_x = g.katec_x AND m.katec_y = g.katec_y
GROUP BY 1, 2,3,4
ORDER BY 5 ASC
"""
# 쿼리 실행 및 결과 보기
df_result = con.sql(query).df()
print("✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:")
zero_stations = df_result[df_result['station_cnt'] == 0]
display(df_result.sort_values(by='station_cnt' , ascending=False))
display(zero_stations)
print(f"총 {len(zero_stations)}개의 좌표가 주유소가 없습니다. (전체 {len(df_result)}개 중)")

✅ 분석 완료! 주유소가 없는 좌표는 다음과 같습니다:


,lat,lon,katec_x,katec_y,station_cnt
499,127.05264,37.57218,316327.374198,552946.873244,58
498,126.71159,37.45515,286026.185393,540318.283437,56
497,126.62632,37.45515,278482.762500,540424.874589,53
496,126.88211,37.51367,301188.181089,546619.539939,52
495,126.71159,37.51367,286115.067144,546812.313298,50
...,...,...,...,...,...
120,126.88211,38.09882,301964.926395,611556.634733,0
119,126.28526,37.27961,247956.140324,521439.023420,0
118,126.20000,37.68921,241264.917221,567036.162257,0
117,126.20000,37.10406,240025.507310,502098.189636,0


,lat,lon,katec_x,katec_y,station_cnt
0,126.45579,37.04555,262654.890017,495205.123242,0
1,126.62632,38.21585,279724.519693,624846.104984,0
2,126.28526,37.92327,249260.952432,592871.629379,0
3,126.37053,38.09882,257098.678156,612220.483541,0
4,126.54106,37.74773,271444.730099,573007.422331,0
...,...,...,...,...,...
177,127.73476,38.27437,376795.465380,630481.772180,0
178,126.20000,36.92852,239656.951628,482618.479689,0
179,127.05264,38.15734,316986.481005,617884.830869,0
180,126.28526,38.09882,249620.141880,612355.187667,0


총 182개의 좌표가 주유소가 없습니다. (전체 500개 중)


In [ ]:
path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
test_df = con.sql(f"""
SELECT uni_cd
, brand_cd
, station_name
, max(case when part_dt=20260127 then 1 else 0 end ) as is_080
, max(case when part_dt=20260128 then 1 else 0 end ) as is_0866

FROM '{path}' 
group by 1,2,3
having max(case when part_dt=20260127 then 1 else 0 end ) = 1
and max(case when part_dt=20260128 then 1 else 0 end ) =0

"""

).df()
    

display(test_df)
    


,uni_cd,brand_cd,station_name,is_080,is_0866
0,A0015150,SKE,원당주유소,1,0
1,A0008727,SOL,천일주유소,1,0
2,A0011215,GSC,㈜월인톨게이트주유소,1,0
3,A0006041,SKE,광명제일주유소 지점,1,0
4,A0008368,SKE,대광주유소,1,0
...,...,...,...,...,...
137,A0007295,ETC,산정주유소,1,0
138,A0003952,GSC,뉴이천주유소,1,0
139,A0007974,SKE,서광유업㈜청도3주유소,1,0
140,A0033018,GSC,(주)보문 북항IC주유소,1,0


In [ ]:
path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
test_df = con.sql(f"""
SELECT uni_cd
, brand_cd
, station_name
, max(case when part_dt=20260126 then 1 else 0 end ) as is_075
, max(case when part_dt=20260127 then 1 else 0 end ) as is_080

FROM '{path}' 
group by 1,2,3
having max(case when part_dt=20260126 then 1 else 0 end ) = 1
and max(case when part_dt=20260127 then 1 else 0 end ) =0

"""

).df()
    

display(test_df)
    


,uni_cd,brand_cd,station_name,is_075,is_080
0,A0003983,GSC,산업로 주유소,1,0
1,A0006214,HDO,㈜석산에너지 도지주유소,1,0
2,A0019295,HDO,HD현대오일뱅크㈜직영 천안현대셀프주유소,1,0
3,A0005963,SOL,오정주유소,1,0
4,A0004399,SKE,송추주유소,1,0
5,A0006269,SOL,신화상사(주)주원고개주유소,1,0
6,A0006594,SKE,달뫼주유소,1,0
7,A0009752,GSC,정인주유소,1,0
8,A0000790,GSC,형제주유소,1,0
9,A0006846,GSC,포동주유소,1,0


In [ ]:
path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
test_df = con.sql(f"""
SELECT *
FROM '{path}' 
where uni_cd='A0033119'
"""

).df()
    

display(test_df)
    


,part_dt,katec_x,katec_y,uni_cd,brand_cd,station_name,station_x,station_y,fuel_type,price,collect_time
0,20260126,323858.736233,552874.352344,A0033119,RTX,대보건설(주) 구리(판교방향)주유소,325659.40973,554484.19308,B027,1639,2026-01-30 23:35:05
1,20260128,328015.030033,557391.756778,A0033119,RTX,대보건설(주) 구리(판교방향)주유소,325659.40973,554484.19308,B027,1639,2026-01-30 23:06:07
2,20260129,326356.097820,551410.221796,A0033119,RTX,대보건설(주) 구리(판교방향)주유소,325659.40973,554484.19308,B027,1639,2026-01-29 22:32:45
3,20260130,326395.783817,555882.256080,A0033119,RTX,대보건설(주) 구리(판교방향)주유소,325659.40973,554484.19308,B027,1639,2026-01-30 22:51:48


In [ ]:
path = "s3://petroleum-project/coord_validation_logs/*/data.parquet"
test_df = con.sql(f"""
SELECT uni_cd
, brand_cd
, station_name
, max(case when part_dt=20260130 then 1 else 0 end ) as is_085
, max(case when part_dt=20260128 then 1 else 0 end ) as is_866

FROM '{path}' 
group by 1,2,3
having max(case when part_dt=20260130 then 1 else 0 end ) = 1
and max(case when part_dt=20260128 then 1 else 0 end ) =0

"""

).df()
    

display(test_df)
    


,uni_cd,brand_cd,station_name,is_085,is_866
0,A0016025,HDO,성환셀프주유소,1,0
1,A0007285,SKE,용샘주유소,1,0
2,A0000215,GSC,마하주유소,1,0
3,A0007840,GSC,무진에너지(주) 한강로셀프주유소,1,0
4,A0032674,SOL,한이에너지(주) 쌍문주유소,1,0
...,...,...,...,...,...
137,A0011145,NHO,문막농협,1,0
138,A0006889,SKE,신안산주유소,1,0
139,A0006858,GSC,통일주유소,1,0
140,A0001557,SKE,계현주유소,1,0
